In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load preprocessed data
with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
y_train = data['y_train']
X_val = data['X_val']
y_val = data['y_val']
X_test = data['X_test']
y_test = data['y_test']
train_indices = data['train_indices']
val_indices = data['val_indices']
train_texts_clean = data['train_texts_clean']
val_texts_clean = data['val_texts_clean']
test_texts_clean = data['test_texts_clean']
vocab = data['vocab']
class_weight_dict = data['class_weight_dict']
max_len = data['max_len']
vocab_size = data['vocab_size']

train_data_clean = pd.read_csv('train_cleaned.csv')
test_data_clean = pd.read_csv('test_cleaned.csv')


print("=" * 70)
print("DATA LOADED")
print("=" * 70)
print(f"\nTrain: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")
print(f"Vocab size: {vocab_size}")
print(f"Max length: {max_len}")
print(f"Classes: 20")

DATA LOADED

Train: (12459, 40)
Val:   (4153, 40)
Test:  (4013, 40)
Vocab size: 5445
Max length: 40
Classes: 20


In [12]:
print("=" * 70)
print("BASELINE MODELS: LOGISTIC REGRESSION & SVM")
print("=" * 70)

# Extract aligned text
X_train_text = data['train_texts_clean']
X_val_text = data['val_texts_clean']
X_test_text = data['test_texts_clean']

print(f"\nData aligned:")
print(f"  Train text: {len(X_train_text)}")
print(f"  Val text: {len(X_val_text)}")
print(f"  Test text: {len(X_test_text)}")

# =========================================================================
# STEP 1: BUILD TF-IDF VECTORIZER
# =========================================================================

print("\n" + "=" * 70)
print("STEP 1: TF-IDF VECTORIZATION")
print("=" * 70)

tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.95,
    ngram_range=(1, 2),
    stop_words='english',
    sublinear_tf=True
)

print("\nFitting TF-IDF on training text...")
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)
X_test_tfidf = tfidf.transform(X_test_text)

print(f"\n✓ TF-IDF complete")
print(f"  Train shape: {X_train_tfidf.shape}")
print(f"  Val shape:   {X_val_tfidf.shape}")
print(f"  Test shape:  {X_test_tfidf.shape}")
print(f"  Vocab size: {len(tfidf.get_feature_names_out())}")

# =========================================================================
# STEP 2: LOGISTIC REGRESSION
# =========================================================================

print("\n" + "=" * 70)
print("STEP 2: LOGISTIC REGRESSION")
print("=" * 70)

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
    solver='lbfgs'
)

print("Training...")
lr_model.fit(X_train_tfidf, y_train)

y_pred_train_lr = lr_model.predict(X_train_tfidf)
y_pred_val_lr = lr_model.predict(X_val_tfidf)
y_pred_test_lr = lr_model.predict(X_test_tfidf)

train_acc_lr = accuracy_score(y_train, y_pred_train_lr)
val_acc_lr = accuracy_score(y_val, y_pred_val_lr)
test_acc_lr = accuracy_score(y_test, y_pred_test_lr)

train_f1_lr = f1_score(y_train, y_pred_train_lr, average='weighted', zero_division=0)
val_f1_lr = f1_score(y_val, y_pred_val_lr, average='weighted', zero_division=0)
test_f1_lr = f1_score(y_test, y_pred_test_lr, average='weighted', zero_division=0)

gap_train_val_lr = train_acc_lr - val_acc_lr
gap_val_test_lr = val_acc_lr - test_acc_lr

print(f"\n✓ Training complete")
print(f"\nAccuracy:")
print(f"  Train: {train_acc_lr:.4f} ({train_acc_lr*100:.2f}%)")
print(f"  Val:   {val_acc_lr:.4f} ({val_acc_lr*100:.2f}%)")
print(f"  Test:  {test_acc_lr:.4f} ({test_acc_lr*100:.2f}%)")

print(f"\nWeighted F1:")
print(f"  Train: {train_f1_lr:.4f}")
print(f"  Val:   {val_f1_lr:.4f}")
print(f"  Test:  {test_f1_lr:.4f}")

print(f"\nGeneralization:")
print(f"  Train-Val gap: {gap_train_val_lr*100:.2f}%")
print(f"  Val-Test gap:  {gap_val_test_lr*100:.2f}%")

# =========================================================================
# STEP 3: SVM (LinearSVC)
# =========================================================================

print("\n" + "=" * 70)
print("STEP 3: SVM (LinearSVC)")
print("=" * 70)

svm_model = LinearSVC(
    max_iter=2000,
    class_weight='balanced',
    random_state=42,
    dual='auto',
    loss='squared_hinge'
)

print("Training (this may take 1-2 minutes)...")
svm_model.fit(X_train_tfidf, y_train)

y_pred_train_svm = svm_model.predict(X_train_tfidf)
y_pred_val_svm = svm_model.predict(X_val_tfidf)
y_pred_test_svm = svm_model.predict(X_test_tfidf)

train_acc_svm = accuracy_score(y_train, y_pred_train_svm)
val_acc_svm = accuracy_score(y_val, y_pred_val_svm)
test_acc_svm = accuracy_score(y_test, y_pred_test_svm)

train_f1_svm = f1_score(y_train, y_pred_train_svm, average='weighted', zero_division=0)
val_f1_svm = f1_score(y_val, y_pred_val_svm, average='weighted', zero_division=0)
test_f1_svm = f1_score(y_test, y_pred_test_svm, average='weighted', zero_division=0)

gap_train_val_svm = train_acc_svm - val_acc_svm
gap_val_test_svm = val_acc_svm - test_acc_svm

print(f"\n✓ Training complete")
print(f"\nAccuracy:")
print(f"  Train: {train_acc_svm:.4f} ({train_acc_svm*100:.2f}%)")
print(f"  Val:   {val_acc_svm:.4f} ({val_acc_svm*100:.2f}%)")
print(f"  Test:  {test_acc_svm:.4f} ({test_acc_svm*100:.2f}%)")

print(f"\nWeighted F1:")
print(f"  Train: {train_f1_svm:.4f}")
print(f"  Val:   {val_f1_svm:.4f}")
print(f"  Test:  {test_f1_svm:.4f}")

print(f"\nGeneralization:")
print(f"  Train-Val gap: {gap_train_val_svm*100:.2f}%")
print(f"  Val-Test gap:  {gap_val_test_svm*100:.2f}%")

BASELINE MODELS: LOGISTIC REGRESSION & SVM

Data aligned:
  Train text: 12459
  Val text: 4153
  Test text: 4013

STEP 1: TF-IDF VECTORIZATION

Fitting TF-IDF on training text...

✓ TF-IDF complete
  Train shape: (12459, 5000)
  Val shape:   (4153, 5000)
  Test shape:  (4013, 5000)
  Vocab size: 5000

STEP 2: LOGISTIC REGRESSION
Training...

✓ Training complete

Accuracy:
  Train: 0.3816 (38.16%)
  Val:   0.0458 (4.58%)
  Test:  0.0399 (3.99%)

Weighted F1:
  Train: 0.3380
  Val:   0.0484
  Test:  0.0390

Generalization:
  Train-Val gap: 33.58%
  Val-Test gap:  0.59%

STEP 3: SVM (LinearSVC)
Training (this may take 1-2 minutes)...

✓ Training complete

Accuracy:
  Train: 0.6644 (66.44%)
  Val:   0.0667 (6.67%)
  Test:  0.0620 (6.20%)

Weighted F1:
  Train: 0.6515
  Val:   0.0755
  Test:  0.0691

Generalization:
  Train-Val gap: 59.77%
  Val-Test gap:  0.47%


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D, 
    Dense, Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping

def build_cnn_model(vocab_size, max_len, embedding_dim=64, num_classes=20):
    """
    Build optimized CNN model for text classification
    
    Architecture:
    ├─ Embedding: vocab_size × embedding_dim
    ├─ Conv1D: 32 filters, kernel=3
    ├─ GlobalMaxPooling1D: Reduce to fixed size
    ├─ Dense: 128 units, ReLU
    ├─ Dropout: 0.2
    └─ Dense: 20 units, Softmax (output)
    
    Total params: ~370K (manageable, no overfitting)
    """
    
    # Input layer
    i = Input(shape=(max_len,))
    
    # Embedding layer (learns word representations)
    x = Embedding(vocab_size, embedding_dim, mask_zero=True)(i)
    
    # Conv1D layer (capture local patterns)
    x = Conv1D(32, kernel_size=3, activation='relu', padding='same')(x)
    
    # Global max pooling (reduce dimension)
    x = GlobalMaxPooling1D()(x)
    
    # Dense layer (learn patterns)
    x = Dense(128, activation='relu')(x)
    
    # Dropout (prevent overfitting)
    x = Dropout(0.2)(x)
    
    # Output layer (20 classes)
    x = Dense(num_classes, activation='softmax')(x)
    
    # Build model
    model = Model(i, x)
    
    return model

# Build model
model = build_cnn_model(vocab_size, max_len)

print("\n" + "=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)
print(model.summary())

# Count parameters
total_params = model.count_params()
print(f"\n✓ Total parameters: {total_params:,}")
print(f"  (Safe threshold: <1M for 12,459 training samples)")
print(f"  Params per sample: {total_params / len(X_train):.1f} (should be <10)")




MODEL ARCHITECTURE
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 40)]              0         
                                                                 
 embedding (Embedding)       (None, 40, 64)            348480    
                                                                 
 conv1d (Conv1D)             (None, 40, 32)            6176      
                                                                 
 global_max_pooling1d (Glob  (None, 32)                0         
 alMaxPooling1D)                                                 
                                                                 
 dense (Dense)               (None, 128)               4224      
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                       

In [4]:
# Compile model
model.compile(
    loss=SparseCategoricalCrossentropy(),
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print("\n" + "=" * 70)
print("MODEL COMPILED")
print("=" * 70)
print(f"Loss function: Sparse Categorical Crossentropy")
print(f"Optimizer: Adam (learning_rate=0.001)")
print(f"Metrics: Accuracy")
print(f"\nReady for training!")


MODEL COMPILED
Loss function: Sparse Categorical Crossentropy
Optimizer: Adam (learning_rate=0.001)
Metrics: Accuracy

Ready for training!


In [5]:
print("\n" + "=" * 70)
print("TRAINING CNN MODEL")
print("=" * 70)

# Define early stopping callback
early_stop = EarlyStopping(
    monitor='val_loss',      # Monitor validation loss
    patience=5,              # Stop if no improvement for 5 epochs
    restore_best_weights=True,  # Keep best weights
    verbose=1
)

# Train model
history = model.fit(
    X_train, y_train,
    epochs=100,              # Max epochs (early stop will trigger earlier)
    batch_size=32,           # Batch size
    validation_data=(X_val, y_val),  # Validation set for monitoring
    class_weight=class_weight_dict,  # Handle imbalance
    callbacks=[early_stop],  # Early stopping
    verbose=1
)

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print(f"Total epochs trained: {len(history.history['loss'])}")
print(f"Final train loss: {history.history['loss'][-1]:.4f}")
print(f"Final train accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final val loss: {history.history['val_loss'][-1]:.4f}")
print(f"Final val accuracy: {history.history['val_accuracy'][-1]:.4f}")


TRAINING CNN MODEL
Epoch 1/100


390/390 [==============================] - 4s 6ms/step - loss: 2.4334 - accuracy: 0.2786 - val_loss: 1.7194 - val_accuracy: 0.5054
Epoch 2/100
390/390 [==============================] - 2s 4ms/step - loss: 1.0684 - accuracy: 0.6661 - val_loss: 1.0470 - val_accuracy: 0.7139
Epoch 3/100
390/390 [==============================] - 2s 4ms/step - loss: 0.4879 - accuracy: 0.8184 - val_loss: 0.9117 - val_accuracy: 0.7481
Epoch 4/100
390/390 [==============================] - 2s 4ms/step - loss: 0.2528 - accuracy: 0.8937 - val_loss: 0.7851 - val_accuracy: 0.7737
Epoch 5/100
390/390 [==============================] - 2s 4ms/step - loss: 0.1361 - accuracy: 0.9437 - val_loss: 0.8003 - val_accuracy: 0.7833
Epoch 6/100
390/390 [==============================] - 2s 4ms/step - loss: 0.0836 - accuracy: 0.9658 - val_loss: 0.8674 - val_accuracy: 0.7785
Epoch 7/100
390/390 [==============================] - 2s 4ms/step - loss: 0.0563 - accuracy: 0.9776 - val_loss: 0.9066 

In [7]:
# =========================================================================
# LSTM MODEL
# =========================================================================

print("\n" + "=*" * 70)
print("BUILDING LSTM MODEL")
print("=" * 70)

from tensorflow.keras.layers import LSTM, Bidirectional

def build_lstm_model(vocab_size, max_len, embedding_dim=64, num_classes=20):
    """
    LSTM model for text classification
    
    Architecture:
    ├─ Embedding: vocab_size × embedding_dim
    ├─ LSTM: 128 units, processes sequence
    ├─ Dropout: 0.2
    ├─ Dense: 64 units, ReLU
    ├─ Dropout: 0.2
    └─ Dense: 20 units, Softmax (output)
    
    Total params: ~400K
    """
    i = Input(shape=(max_len,))
    x = Embedding(vocab_size, embedding_dim, mask_zero=True)(i)
    x = LSTM(128, return_sequences=False)(x)
    x = Dropout(0.2)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(num_classes, activation='softmax')(x)
    model = Model(i, x)
    return model

# Build model
lstm_model = build_lstm_model(vocab_size, max_len)

print(lstm_model.summary())
print(f"\n✓ Total parameters: {lstm_model.count_params():,}")

# Compile
lstm_model.compile(
    loss=SparseCategoricalCrossentropy(),
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print("\n✓ LSTM model compiled")

# Train
print("\nTraining LSTM (this may take 2-3 minutes per epoch)...")
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

print("\n✓ LSTM training complete")

# =========================================================================
# BILSTM MODEL
# =========================================================================

print("\n" + "=" * 70)
print("BUILDING BILSTM MODEL")
print("=" * 70)

def build_bilstm_model(vocab_size, max_len, embedding_dim=64, num_classes=20):
    """
    Bidirectional LSTM model for text classification
    
    Architecture:
    ├─ Embedding: vocab_size × embedding_dim
    ├─ BiLSTM: 128 units (forward + backward), processes sequence bidirectionally
    ├─ Dropout: 0.2
    ├─ Dense: 64 units, ReLU
    ├─ Dropout: 0.2
    └─ Dense: 20 units, Softmax (output)
    
    Total params: ~500K (more because BiLSTM has 2 directions)
    """
    i = Input(shape=(max_len,))
    x = Embedding(vocab_size, embedding_dim, mask_zero=True)(i)
    x = Bidirectional(LSTM(128, return_sequences=False))(x)
    x = Dropout(0.2)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(num_classes, activation='softmax')(x)
    model = Model(i, x)
    return model

# Build model
bilstm_model = build_bilstm_model(vocab_size, max_len)

print(bilstm_model.summary())
print(f"\n✓ Total parameters: {bilstm_model.count_params():,}")

# Compile
bilstm_model.compile(
    loss=SparseCategoricalCrossentropy(),
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print("\n✓ BiLSTM model compiled")

# Train
print("\nTraining BiLSTM (this may take 3-4 minutes per epoch)...")
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

bilstm_history = bilstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

print("\n✓ BiLSTM training complete")


=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*
BUILDING LSTM MODEL
Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 40)]              0         
                                                                 
 embedding_1 (Embedding)     (None, 40, 64)            348480    
                                                                 
 lstm (LSTM)                 (None, 128)               98816     
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense_2 (Dense)             (None, 64)                8256      
                                                                 
 dropout_2 (Dropout)         

In [8]:
# =========================================================================
# EVALUATION: GET PREDICTIONS FOR ALL 3 MODELS
# =========================================================================

print("\n" + "=" * 70)
print("EVALUATING ALL DEEP LEARNING MODELS")
print("=" * 70)

# Get predictions from all models
print("\nGenerating predictions...")

# CNN (already have)
y_pred_cnn_train = np.argmax(model.predict(X_train, verbose=0), axis=1)
y_pred_cnn_val = np.argmax(model.predict(X_val, verbose=0), axis=1)
y_pred_cnn_test = np.argmax(model.predict(X_test, verbose=0), axis=1)

# LSTM
y_pred_lstm_train = np.argmax(lstm_model.predict(X_train, verbose=0), axis=1)
y_pred_lstm_val = np.argmax(lstm_model.predict(X_val, verbose=0), axis=1)
y_pred_lstm_test = np.argmax(lstm_model.predict(X_test, verbose=0), axis=1)

# BiLSTM
y_pred_bilstm_train = np.argmax(bilstm_model.predict(X_train, verbose=0), axis=1)
y_pred_bilstm_val = np.argmax(bilstm_model.predict(X_val, verbose=0), axis=1)
y_pred_bilstm_test = np.argmax(bilstm_model.predict(X_test, verbose=0), axis=1)

print("✓ Predictions generated for all models")

# =========================================================================
# CALCULATE METRICS FOR ALL MODELS
# =========================================================================

def get_metrics(y_true_train, y_pred_train, y_true_val, y_pred_val, y_true_test, y_pred_test, model_name):
    """Calculate and display metrics for a model"""
    train_acc = accuracy_score(y_true_train, y_pred_train)
    val_acc = accuracy_score(y_true_val, y_pred_val)
    test_acc = accuracy_score(y_true_test, y_pred_test)
    
    train_f1 = f1_score(y_true_train, y_pred_train, average='weighted', zero_division=0)
    val_f1 = f1_score(y_true_val, y_pred_val, average='weighted', zero_division=0)
    test_f1 = f1_score(y_true_test, y_pred_test, average='weighted', zero_division=0)
    
    gap_train_val = train_acc - val_acc
    gap_val_test = val_acc - test_acc
    
    return {
        'model': model_name,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'test_acc': test_acc,
        'train_f1': train_f1,
        'val_f1': val_f1,
        'test_f1': test_f1,
        'gap_train_val': gap_train_val,
        'gap_val_test': gap_val_test,
        'y_pred_test': y_pred_test
    }

# Get metrics for all DL models
cnn_metrics = get_metrics(y_train, y_pred_cnn_train, y_val, y_pred_cnn_val, y_test, y_pred_cnn_test, "CNN")
lstm_metrics = get_metrics(y_train, y_pred_lstm_train, y_val, y_pred_lstm_val, y_test, y_pred_lstm_test, "LSTM")
bilstm_metrics = get_metrics(y_train, y_pred_bilstm_train, y_val, y_pred_bilstm_val, y_test, y_pred_bilstm_test, "BiLSTM")

# Display comparison
print("\n" + "=" * 70)
print("DEEP LEARNING MODELS COMPARISON")
print("=" * 70)

comparison_data = [cnn_metrics, lstm_metrics, bilstm_metrics]

print(f"\n{'Model':<10} {'Train Acc':<12} {'Val Acc':<12} {'Test Acc':<12} {'Train-Val Gap':<15}")
print("-" * 65)
for m in comparison_data:
    print(f"{m['model']:<10} {m['train_acc']:<12.4f} {m['val_acc']:<12.4f} {m['test_acc']:<12.4f} {m['gap_train_val']:<15.4f}")

print(f"\n{'Model':<10} {'Train F1':<12} {'Val F1':<12} {'Test F1':<12}")
print("-" * 50)
for m in comparison_data:
    print(f"{m['model']:<10} {m['train_f1']:<12.4f} {m['val_f1']:<12.4f} {m['test_f1']:<12.4f}")

# Best model
best_model = max(comparison_data, key=lambda x: x['test_acc'])
print(f"\n✓ BEST MODEL: {best_model['model']} (Test Acc: {best_model['test_acc']:.4f})")


EVALUATING ALL DEEP LEARNING MODELS

Generating predictions...
✓ Predictions generated for all models

DEEP LEARNING MODELS COMPARISON

Model      Train Acc    Val Acc      Test Acc     Train-Val Gap  
-----------------------------------------------------------------
CNN        0.9567       0.7737       0.7745       0.1830         
LSTM       0.9329       0.7558       0.7376       0.1771         
BiLSTM     0.9429       0.7790       0.7752       0.1640         

Model      Train F1     Val F1       Test F1     
--------------------------------------------------
CNN        0.9566       0.7760       0.7764      
LSTM       0.9330       0.7588       0.7409      
BiLSTM     0.9429       0.7810       0.7774      

✓ BEST MODEL: BiLSTM (Test Acc: 0.7752)


In [9]:
# Save all models
print("\nSaving models...")
lstm_model.save('lstm_model.h5')
bilstm_model.save('bilstm_model.h5')

# Save metrics
metrics_summary = {
    'cnn': cnn_metrics,
    'lstm': lstm_metrics,
    'bilstm': bilstm_metrics
}

with open('dl_models_metrics.pkl', 'wb') as f:
    pickle.dump(metrics_summary, f)

print("✓ All models saved")


Saving models...
✓ All models saved
